# DADA-2000 original — Phase 2 / build the **T2** corpus  (+ Phase 3 / **Gate W**)

Plan: `.project/plans/katvad-dada-original-phase2-t2.md` · Parent: `.project/plans/katvad-dada-original-corpus.md`
Gate D0 passed on 2026-09-15 (`auc_macro` **0.6518**), which is what licenses this phase.

**What comes out:** a windowed corpus at `$KATVAD_DATA_ROOT/DADA2000_orig` (W=16, hop 8,
negatives cut from *inside* the accident videos) and one Gate-W verdict.

| Gate W | bar |
|---|---|
| length leak (clip AUC) | ≤ 0.55 |
| clip oracle (micro) | ≤ 0.75 |
| abnormal source retention | ≥ 0.90 |
| class ratio | within 1:3 either way |
| kernel coverage (median, kernel **3**) | ≤ 0.35 |
| two-class test clips | ≥ 300 |

**Any miss → STOP.** Re-open plan §4 and the parent plan's §5.2 table; do not train on a corpus that failed.

**GPU runtime.** The expensive part is I/O, not compute: `images` is **94.01 GiB** over six
spanned-zip volumes, so the extraction is **sharded** — extract → encode → delete → next.
Phase 1 already cached 400 of these clips; the loop skips them.

> **Rebuilding at a different geometry costs no extraction.** The CLIP cache is keyed by
> **source clip** and a window is a slice, so changing `T2_WINDOW_LENGTH` re-runs only
> §3 → §6 (~2 minutes, no GPU). §2 is needed once, ever.
>
> **2026-09-16 — Gate W FAILED at the package default W=16 / hop 8:** clip oracle **0.7529**
> against the 0.75 bar (`F_norm` 6,528 > `X` 6,377 by 151 frames = 0.77 % of the test set).
> §0 now overrides to **W=20 / hop 8**, where the same census gives oracle **0.7037**,
> retention 0.957, 798 two-class windows — and the EDA's CRITICAL *"micro AUC is mostly clip
> classification"* verdict disappears. `core/constants.py` keeps 16 until a geometry passes.

> **Nothing here restates the pipeline's own values.** Paths, the T2 window geometry, the
> dataset name and the four dataset-file names all come from `core.constants` /
> `core.data.dataset_files`, so a rename in the package cannot leave this notebook building a
> second corpus beside the real one (lesson **C2**). The one number that is *not* in the
> package is `score_head_kernel = 3` — §5 says why, and Phase 4 must pass it explicitly.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_CKPT_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), (
    f'no checkout at {REPO} -- clone/pull the repo there before running anything below')

# `python -m core...` puts the CWD on sys.path, but a %%bash cd does not survive and
# the repo is not pip-installed on Colab. PYTHONPATH is what makes the CHILD processes
# below importable; sys.path is what makes THIS kernel importable.
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

# core.constants reads KATVAD_* at import time, so this import must come after the
# block above -- and every path below is then the one `core` itself computes. The
# --out-dir / --output-dir passed later are these same values, written explicitly so
# the run manifest records them (lesson C17).
from core import constants  # noqa: E402

DATASET = constants.DADA_ORIGIN_DATASET                      # 'DADA2000_orig'
os.environ['DADA_ORIG'] = str(
    constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME)
os.environ['P2'] = '/content/p2'          # VM-local NVMe, never Drive
os.makedirs(os.environ['P2'], exist_ok=True)

# The T2 corpus itself is a few MB of JSON -- write it straight to Drive so a
# recycled runtime cannot take it with it (lesson C17; Phase 1 lost its census).
os.environ['T2'] = str(constants.DATA_ROOT / DATASET)          # == --out-dir's default
# Features are keyed by SOURCE CLIP and a window is a slice, so ONE full-clip cache
# serves Phase 1 and Phase 2 alike -- and Phase 1's 400 .npy are reused. Never
# cache/clip/DADA2000: that is the trimmed archive (lesson C2).
os.environ['CACHE'] = str(constants.CLIP_CACHE_DIR / DATASET)  # == extract's default
# --- T2 geometry -------------------------------------------------------------
# Gate W IS the experiment that varies these, so they live here, in ONE place,
# printed below and recorded in the manifest (C17). core/constants.py is updated
# only AFTER a geometry passes Gate W, never before -- which is why the package
# default is still 16 and this notebook overrides it.
#
# 2026-09-16, measured: W=16 / hop 8 (the package default) FAILED Gate W with a
# clip oracle of 0.7529 against the 0.75 bar -- F_norm 6,528 > X 6,377 by 151
# frames, 0.77 % of the test set, in C33's closed form. Re-measured on the REAL
# archive census, W=20 / hop 8 gives oracle 0.7037, retention 0.957, 798 two-class
# windows, ratio 2.80 -- and the EDA's CRITICAL "micro AUC is mostly clip
# classification" verdict disappears entirely. The plan's predicted 0.6631 at
# W=16 is refuted; it belongs to a W of about 24, which fails retention instead.
T2_WINDOW_LENGTH = 20                                    # package default 16 -- FAILED
T2_WINDOW_STRIDE = constants.DADA_ORIGIN_WINDOW_STRIDE   # 8
T2_MAX_PER_CLIP = constants.WINDOW_MAX_PER_CLIP          # 4
T2_MIN_POSITIVE = constants.WINDOW_MIN_POSITIVE          # 1
T2_SCORE_HEAD_KERNEL = 3       # C27: 3 of 20 sampled frames = 15 % of the window

# Stamped with the geometry, so a rebuild cannot overwrite the record of the
# build it replaced -- the W=16 Gate-W run stays readable beside this one.
os.environ['EDA'] = str(
    constants.OUTPUT_ROOT / 'EDA'
    / f'{DATASET}_T2_w{T2_WINDOW_LENGTH}s{T2_WINDOW_STRIDE}')
# The per-shard frame census is the ONE artifact that cannot be rebuilt by re-running
# anything cheap: after a shard's frames are deleted it is the only record of how many
# images each clip had. Phase 1 kept it on /content and lost it to a recycled runtime;
# this notebook did the same on 2026-09-16 and section 3 failed with "No census file
# matched". It lives on Drive now -- and section 2.0 can rebuild it from the zip index.
os.environ['COUNTS'] = f"{os.environ['T2']}/counts"
os.makedirs(os.environ['COUNTS'], exist_ok=True)

print(f'dataset   {DATASET}')
print(f'geometry  W={T2_WINDOW_LENGTH} hop={T2_WINDOW_STRIDE} cap={T2_MAX_PER_CLIP} '
      f'min_pos={T2_MIN_POSITIVE} kernel={T2_SCORE_HEAD_KERNEL}'
      + ('  <-- OVERRIDES core/constants.py '
         f'(W={constants.DADA_ORIGIN_WINDOW_LENGTH})'
         if T2_WINDOW_LENGTH != constants.DADA_ORIGIN_WINDOW_LENGTH else ''))
print('\n'.join(f'{k:10s} {os.environ[k]}'
                for k in ('REPO', 'DADA_ORIG', 'T2', 'COUNTS', 'CACHE', 'EDA', 'P2')))

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
# core/data/video_io.py imports `av` at module import, and core.tools.extract_clip_features
# imports video_io -- so `av` is needed even though this corpus ships PNG frames, not videos.
pip install -q av
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU'
df -h /content | tail -1      # section 2 sizes its shards against this

## 1. Preflight — refuse to start on a tree or an archive that is not what this notebook expects

Every assertion here failed at least once during Phases 0/1. They cost seconds; the run they
guard costs hours.

In [ ]:
import json
import math
import shutil
import subprocess

ORIG  = Path(os.environ['DADA_ORIG'])
P2    = Path(os.environ['P2'])
CACHE = Path(os.environ['CACHE'])
ENV   = dict(os.environ)      # PYTHONPATH already set in section 0

# --- 1.1 the tree carries Phase 2 code at all -----------------------------
from core.data import dada_origin  # noqa: E402
from core.data import dataset_files as ds_files  # noqa: E402

assert hasattr(dada_origin, 'census_pass'), (
    'This checkout predates Phase 2. Pull the branch that ships '
    'core/data/dada_origin.py before running anything below.')

# --- 1.2 the annotation parses to the corpus we think it is ---------------
ANNOTATION = ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME
assert ANNOTATION.is_file(), f'missing {ANNOTATION}'
rows = dada_origin.parse_annotation(ANNOTATION)
types = {r.type_id for r in rows}
N_CLIPS = len(rows)

# Rows are the workbook's 1,962 filtered to those flagged as an accident with a window
# inside [0, total] -- Phase 0 measured 1,945 of them (DADA_ORIGIN_PHASE0.md section 3).
print(f'annotation: {N_CLIPS} accident clips over {len(types)} types')
assert len(types) == 52, f'{len(types)} types, expected 52 -- wrong workbook?'
assert len({r.video_id for r in rows}) == N_CLIPS, 'duplicate (type, video)'
if N_CLIPS != 1945:
    print(f'NOTE: Phase 0 measured 1945 accident rows, this file parses {N_CLIPS}. '
          'Not fatal -- but the workbook is not the one every number below was sized on.')

# Clips grouped by accident type, which is both the shard unit (section 2) and what
# dada_origin.split_by_type stratifies the train/test split on.
BY_TYPE: dict[int, list] = {}
for r in rows:
    BY_TYPE.setdefault(r.type_id, []).append(r)
for t in BY_TYPE:
    BY_TYPE[t].sort(key=lambda r: r.video)
sizes = sorted(len(g) for g in BY_TYPE.values())
print(f'clips per type: min {sizes[0]}, median {sizes[len(sizes) // 2]}, max {sizes[-1]}')

# --- 1.3 every zip volume is present -------------------------------------
vols = sorted(ORIG.glob(f'{constants.DADA_ORIGIN_ROOT_DIRNAME}.z*'))
total = sum(v.stat().st_size for v in vols) / 2**30
print(f'\narchive: {len(vols)} volumes, {total:.1f} GiB')
for v in vols:
    print(f'   {v.name:20s} {v.stat().st_size / 2**30:6.2f} GiB')
assert len(vols) >= 2, 'a spanned archive needs its .z01.. volumes beside the .zip'

# --- 1.4 what Phase 1 already cached -------------------------------------
CACHE.mkdir(parents=True, exist_ok=True)
cached = {p.stem for p in CACHE.glob('*.npy')}
print(f'\ncache {CACHE}: {len(cached)} clips already encoded '
      f'({len(cached & {r.video_id for r in rows})} of them in this corpus)')
print(f'/content: {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')

## 2.0 Archive census from the zip **index** — no extraction, no GPU

`7z l -slt` reads the archive's central directory and lists every path without
unpacking a byte. That gives an exact per-clip image count for all 1,945 clips in one pass.

Two jobs:
* **preflight** — confirms the archive holds what the annotation claims, before hours of I/O;
* **recovery** — if the per-shard censuses are ever lost (VM recycled, `/content` wiped) this
  rebuilds them from the archive, so a lost census never costs a re-extraction.

The shard censuses stay authoritative where they exist: they record what the extractor actually
read. This one records what the archive *contains*. Section 3 merges them in that order.

In [ ]:
import re

ZIP_CENSUS = Path(os.environ['COUNTS']) / 'archive_index.json'
ARCHIVE = f'{constants.DADA_ORIGIN_ROOT_DIRNAME}.zip'

if ZIP_CENSUS.exists():
    zip_counts = json.loads(ZIP_CENSUS.read_text())
    print(f'reusing {ZIP_CENSUS.name}: {len(zip_counts)} clips')
else:
    print('listing the archive index (no extraction; takes a minute) ...')
    proc = subprocess.run(['7z', 'l', '-slt', '-ba', ARCHIVE],
                          cwd=str(ORIG), capture_output=True, text=True)
    if proc.returncode:
        print(proc.stdout[-2000:], proc.stderr[-2000:], sep='\n')
        raise RuntimeError(f'7z l failed with exit {proc.returncode}')

    # {ROOT}/{type}/{video:03d}/{images}/{frame}.png -- the layout measured in
    # DADA_ORIGIN_PHASE0.md section 3.1, spelled from the same constants the
    # resolver in core.data.dada_origin.clip_folder uses.
    entry = re.compile(
        rf'^{constants.DADA_ORIGIN_ROOT_DIRNAME}/(\d+)/(\d+)/'
        rf'{constants.DADA_ORIGIN_IMAGES_SUBDIR}/[^/]+\.(png|jpg|jpeg)$', re.I)
    tally: dict[str, int] = {}
    for line in proc.stdout.splitlines():
        if not line.startswith('Path = '):
            continue
        m = entry.match(line[7:].strip().replace('\\', '/'))
        if m:
            vid = dada_origin.video_id(int(m.group(1)), int(m.group(2)))
            tally[vid] = tally.get(vid, 0) + 1
    if not tally:
        raise RuntimeError(
            f'7z listed the archive but no path matched {entry.pattern} -- '
            'check the layout with: !cd "$DADA_ORIG" && 7z l -ba '
            f'{ARCHIVE} | head -20')
    zip_counts = tally
    ZIP_CENSUS.write_text(json.dumps(zip_counts, indent=2, sort_keys=True), encoding='utf-8')
    print(f'wrote {ZIP_CENSUS} ({len(zip_counts)} clips)')

annotated = {r.video_id for r in rows}
covered = annotated & zip_counts.keys()
print(f'archive covers {len(covered)}/{len(annotated)} annotated clips')
frames_total = sum(zip_counts[v] for v in covered)
print(f'frames in those clips: {frames_total:,}  (median per clip '
      f'{sorted(zip_counts[v] for v in covered)[len(covered) // 2]})')
NOT_IN_ARCHIVE = annotated - zip_counts.keys()
if NOT_IN_ARCHIVE:
    print(f'WARNING: {len(NOT_IN_ARCHIVE)} annotated clips are NOT in the archive, '
          f'e.g. {sorted(NOT_IN_ARCHIVE)[:5]}')
    print('        Section 2 treats exactly these as expected-absent; anything else '
          'missing after a shard is an extraction failure and raises.')

## 2. Sharded extraction — extract → census + farm → encode → **delete**

`images` is 94 GiB; no runtime holds it. One shard of `SHARD_CLIPS` clips is ~`SHARD_CLIPS × 50 MiB`.

**Shards are packed out of whole accident-type directories.** That is the taxonomy
`dada_origin.split_by_type` stratifies on, so an interrupted run leaves whole types finished
rather than a slice of every type — and the include patterns stay grouped per directory. The
patterns themselves keep the **per-clip** form measured in `DADA_ORIGIN_PHASE0.md` §5
(`.../{video:03d}/images/*`, where dropping `/images/` pulls 5.3× the bytes); lesson **C20**'s
hazard is a *shell* argument list, and these go to `subprocess` as a Python list, bounded by
`SHARD_CLIPS`.

Four things this loop does that the Phase 1 loop had to learn the hard way:
* it **asserts its own input** before starting (a stale intermediate silently ran 52 clips once);
* it **skips** a shard already encoded, so a disconnect costs one shard, not the run;
* it **verifies each shard's census** against the archive index and raises on a clip that
  should have been there — a silent short extraction is what makes retention fail in §5 for a
  reason that has nothing to do with the corpus (lesson **C10**);
* the symlink farm is built by `--census-only --flat-frames-dir`, linking at the **`images`**
  directory itself — `pathlib` will not recurse into a symlinked directory (lesson **C26**).

In [ ]:
SHARD_CLIPS = 150     # ~7.5 GiB of PNGs per shard; lower it if section 1.4 showed less disk

# Drive, not /content: a census on the VM dies with the VM, and section 3 then
# cannot build the corpus even though every feature file survived (C17).
counts_dir = Path(os.environ['COUNTS'])
counts_dir.mkdir(parents=True, exist_ok=True)


def free_gib() -> float:
    return shutil.disk_usage('/content').free / 2**30


def run(cmd, cwd, capture=False):
    """Run a child and make its failure legible, not a bare CalledProcessError."""
    proc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=ENV,
                          capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}'
                           + ('' if capture else '  -- scroll up for the child output'))
    if capture and proc.stdout:
        print(proc.stdout.rstrip())


def plan_shards(by_type: dict, budget: int) -> list[list]:
    """Pack whole TYPE directories into shards of at most `budget` clips.

    A type bigger than the budget is cut into EVEN chunks (not budget-sized ones), so
    a 166-clip type becomes 83+83 and can share a shard instead of leaving a 16-clip
    runt behind -- a runt costs a whole extra pass over the archive's central directory.
    """
    plan: list[list] = []
    bucket: list = []
    for type_id in sorted(by_type):
        group = by_type[type_id]
        n_chunks = math.ceil(len(group) / budget)
        for i in range(n_chunks):
            chunk = group[i * len(group) // n_chunks:(i + 1) * len(group) // n_chunks]
            if bucket and len(bucket) + len(chunk) > budget:
                plan.append(bucket)
                bucket = []
            bucket += chunk
    if bucket:
        plan.append(bucket)
    return plan


assert len(rows) == N_CLIPS, 'section 1 did not run in this session'
SHARDS = plan_shards(BY_TYPE, SHARD_CLIPS)
print(f'{N_CLIPS} clips -> {len(SHARDS)} shards of at most {SHARD_CLIPS} '
      f'(~{SHARD_CLIPS * 50 / 1024:.1f} GiB each)')
# Every shard is one more pass over a 94 GiB spanned archive on a FUSE mount, so the
# price of keeping shards inside type boundaries is worth seeing before paying it.
floor_passes = math.ceil(N_CLIPS / SHARD_CLIPS)
print(f'type-blind packing would need {floor_passes} passes; type alignment costs '
      f'{len(SHARDS) - floor_passes} more. Raise SHARD_CLIPS if section 1.4 showed the disk '
      f'for it (~{SHARD_CLIPS * 50 / 1024:.1f} GiB of PNGs per pass).')
for i, shard in enumerate(SHARDS):
    ts = sorted({r.type_id for r in shard})
    print(f'  {i:03d}: {len(shard):3d} clips, types {ts[0]}..{ts[-1]}' if len(ts) > 1
          else f'  {i:03d}: {len(shard):3d} clips, type {ts[0]}')

In [ ]:
IMAGES = constants.DADA_ORIGIN_IMAGES_SUBDIR
ROOT   = constants.DADA_ORIGIN_ROOT_DIRNAME

# The archive index carries the SAME per-clip counts a shard census does -- it is
# what section 3 falls back to. Holding it here lets an already-encoded shard whose
# census was lost be skipped AND repaired, instead of re-extracting 7.5 GiB to
# rediscover a number that is already on disk.
zip_census = json.loads(ZIP_CENSUS.read_text()) if ZIP_CENSUS.exists() else {}

for index, shard in enumerate(SHARDS):
    tag = f'{index:03d}'
    census = counts_dir / f'{tag}.json'
    encoded_all = all((CACHE / f'{r.video_id}.npy').exists() for r in shard)

    if encoded_all and census.exists():
        print(f'shard {tag}: already encoded, skipping')
        continue
    if encoded_all and all(r.video_id in zip_census or r.video_id in NOT_IN_ARCHIVE
                           for r in shard):
        # Every clip is encoded and every count is known -- re-extracting would only
        # rewrite a census the archive index already answers (lesson C17: the record
        # was the thing that went missing, not the data).
        payload = {r.video_id: zip_census[r.video_id] for r in shard if r.video_id in zip_census}
        census.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')
        print(f'shard {tag}: already encoded, census rebuilt from the archive index '
              f'({len(payload)} clips)')
        continue

    frames, farm = P2 / 'frames', P2 / f'farm_{tag}'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    # Per-clip patterns, the form measured in DADA_ORIGIN_PHASE0.md section 5:
    # WITHOUT the /images/ component 7z pulls five sibling subdirs and 5.3x the bytes.
    patterns = [f'{ROOT}/{r.type_id}/{r.video:03d}/{IMAGES}/*' for r in shard]

    print(f'--- shard {tag}: {len(shard)} clips, '
          f'types {sorted({r.type_id for r in shard})}, {free_gib():.1f} GiB free')
    run(['7z', 'x', f'{ROOT}.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns], cwd=ORIG)
    run([sys.executable, '-m', 'core.data.dada_origin',
         '--annotation', ANNOTATION, '--frames-dir', frames,
         '--counts-out', census, '--flat-frames-dir', farm, '--census-only'],
        cwd=REPO, capture=True)

    # A short extraction is silent otherwise: the census simply omits the clip, the
    # corpus is built without it, and section 5 reads a retention miss that has nothing
    # to do with the construction (C10).
    got = json.loads(census.read_text())
    missing = [r.video_id for r in shard if r.video_id not in got]
    unexpected = [v for v in missing if v not in NOT_IN_ARCHIVE]
    if unexpected:
        raise RuntimeError(
            f'shard {tag}: {len(unexpected)} clips are in the archive index but produced no '
            f'readable frames: {sorted(unexpected)[:5]}. Re-run this shard; do NOT proceed.')
    if missing:
        print(f'    {len(missing)} clips absent from the archive itself, as section 2.0 said')

    run([sys.executable, '-m', 'core.tools.extract_clip_features',
         '--frames-dir', farm, '--dataset', DATASET,
         '--stride', constants.FRAME_STRIDE, '--batch-size', '32', '--device', 'auto',
         '--no-center-crop', '--output-dir', CACHE], cwd=REPO)

    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    cache: {len(list(CACHE.glob("*.npy")))} clips, {free_gib():.1f} GiB free')

censuses = sorted(p for p in counts_dir.glob('*.json') if p.name != ZIP_CENSUS.name)
merged = {}
for c in censuses:
    merged.update(json.loads(c.read_text()))
encoded = {p.stem for p in CACHE.glob('*.npy')}
print(f'\nshards done: {len(censuses)}/{len(SHARDS)} | census: {len(merged)} clips | '
      f'cache: {len(encoded)} .npy')
missing = {r.video_id for r in rows} - encoded
if missing:
    print(f'WARNING: {len(missing)} annotated clips have no feature file, e.g. '
          f'{sorted(missing)[:5]}')

## 3. Build the T2 corpus from the merged censuses — no frames needed

The frames are gone by now; the on-disk counts survive in `counts/*.json`, which is exactly
what `--counts-file` consumes. Written straight to Drive.

Every geometry flag below is `core.constants`' own value — `--window-length`,
`--window-stride`, `--window-max-per-clip`, `--window-min-positive`, `--test-ratio`,
`--stride` and `--seed` are all the CLI's defaults, passed explicitly so the run manifest in
§6 records what this build actually was (lesson **C17**).

In [ ]:
T2 = Path(os.environ['T2'])
counts_dir = Path(os.environ['COUNTS'])
ZIP_CENSUS = counts_dir / 'archive_index.json'
RESOLVED = counts_dir / 'resolved_census.json'

# Merge order = trust order. The archive index says what the ZIP holds; a shard
# census says what the extractor actually read. The latter wins where it exists.
census: dict[str, int] = {}
if ZIP_CENSUS.exists():
    census.update(json.loads(ZIP_CENSUS.read_text()))
shards = sorted(p for p in counts_dir.glob('*.json')
                if p.name not in {ZIP_CENSUS.name, RESOLVED.name})
for c in shards:
    census.update(json.loads(c.read_text()))

if not census:
    raise SystemExit(
        f'No census under {counts_dir}.\n'
        '  - If section 2 has not run in this runtime: run section 2.0, then section 2.\n'
        '  - If the runtime was recycled: section 2.0 alone rebuilds the census from the\n'
        '    archive index without extracting anything, and the CLIP cache on Drive is\n'
        '    untouched -- check it with len(list(CACHE.glob("*.npy"))).')

# A clip with labels but no feature file fails much later, inside the EDA's
# feature section (core/eda/features.py:346). Intersect here, loudly, and hand
# dada_origin ONE resolved file instead of a glob.
encoded = {p.stem for p in CACHE.glob('*.npy')}
usable = {v: n for v, n in census.items() if v in encoded}
dropped = len(census) - len(usable)
print(f'census {len(census)} clips | encoded {len(encoded)} | usable {len(usable)}'
      + (f' | dropped {dropped} with no .npy' if dropped else ''))
if not usable:
    raise SystemExit('No clip has BOTH a census entry and a feature file; run section 2.')
if len(usable) < 0.9 * N_CLIPS:
    print(f'WARNING: only {len(usable)}/{N_CLIPS} annotated clips are usable, i.e. below '
          "Gate W's 0.90 retention bar BEFORE windowing has dropped anything. Section 2 is "
          'incomplete -- finish it, or §5 will fail for the wrong reason.')

RESOLVED.write_text(json.dumps(usable, indent=2, sort_keys=True), encoding='utf-8')

# The corpus is rebuilt IN PLACE: the four dataset files plus windows.json are
# overwritten. That is cheap and correct -- the CLIP cache is keyed by SOURCE clip
# and a window is a slice, so changing the geometry costs no re-extraction -- but
# it does mean the data dir holds exactly one geometry at a time. Which one is
# recorded in meta.json / windows.json, and in the manifest in section 6.
if (T2 / constants.WINDOWS_FILENAME).is_file():
    previous = json.loads((T2 / constants.WINDOWS_FILENAME).read_text())
    was = {w['end'] - w['start'] for w in previous.values()}
    print(f'overwriting an existing build in {T2}: {len(previous)} windows of '
          f'length {sorted(was)} -> W={T2_WINDOW_LENGTH} hop={T2_WINDOW_STRIDE}')

run([sys.executable, '-m', 'core.data.dada_origin',
     '--annotation', ANNOTATION,
     '--counts-file', RESOLVED,
     '--out-dir', T2,
     '--stride', constants.FRAME_STRIDE,
     '--window-length', T2_WINDOW_LENGTH,
     '--window-stride', T2_WINDOW_STRIDE,
     '--window-max-per-clip', T2_MAX_PER_CLIP,
     '--window-min-positive', T2_MIN_POSITIVE,
     '--test-ratio', constants.DADA_ORIGIN_TEST_RATIO,
     '--seed', constants.SEED,
     '--allow-missing-frames'], cwd=REPO, capture=True)
print('\n', sorted(p.name for p in T2.iterdir()))

## 4. Corpus sanity — assert the construction before measuring it

Gate W measures the corpus; this cell checks it **is** the corpus. A build that quietly
produced 32-frame windows, put one accident in both splits, or wrote id files that disagree
with the split would still produce a plausible-looking EDA report.

`train_ids.txt` / `test_ids.txt` hold **source** ids, not window ids — they are what Phase 4
feeds to `extract_clip_features --ids-file` and `raft_extract --ids-file`, so they are checked
here against both the split and the feature cache.

In [ ]:
windows = json.loads((T2 / constants.WINDOWS_FILENAME).read_text())
meta    = json.loads((T2 / constants.META_FILENAME).read_text())
labels  = json.loads((T2 / constants.LABELS_TRAIN_FILENAME).read_text())
frame_test = json.loads((T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())
defs    = json.loads((T2 / constants.DEFS_FILENAME).read_text())

W = T2_WINDOW_LENGTH
lengths = {w['end'] - w['start'] for w in windows.values()}
assert lengths == {W}, f'window lengths are {sorted(lengths)}, expected exactly {W}'

by_split = {'train': set(), 'test': set()}
for wid, entry in meta.items():
    by_split[entry['split']].add(windows[wid]['source'])
shared = by_split['train'] & by_split['test']
assert not shared, f'{len(shared)} source clips are in BOTH splits: {sorted(shared)[:3]}'

assert set(labels.values()) == {0, 1}, (
    f'train windows carry labels {sorted(set(labels.values()))}; DVSFeatureDataset '
    'needs both classes -- T2 gets its negatives from the accident videos themselves')
assert all(e['source_is_abnormal'] for e in meta.values()), 'a normal SOURCE clip leaked in'

for key in ('texts', 'causes', 'measures', 'weather', 'light', 'scenes', 'linear'):
    assert key in next(iter(meta.values())), (
        f'meta.json is missing {key} -- the workbook header for it did not match '
        'core.data.dada_origin.META_COLUMNS, and the column was dropped silently')

# train_ids.txt / test_ids.txt: SOURCE ids, written by core.data.dada.write_windowed.
ids = {}
for split, filename in (('train', ds_files.TRAIN_IDS_FILENAME),
                        ('test', ds_files.TEST_IDS_FILENAME)):
    ids[split] = set((T2 / filename).read_text(encoding='utf-8').split())
    assert ids[split] == by_split[split], (
        f'{filename} lists {len(ids[split])} ids but meta.json puts '
        f'{len(by_split[split])} sources in {split}')
uncached = (ids['train'] | ids['test']) - {p.stem for p in CACHE.glob('*.npy')}
assert not uncached, (
    f'{len(uncached)} source ids have no .npy in {CACHE}: {sorted(uncached)[:5]} -- '
    'section 3 should have filtered these; the cache changed under the build')

n_pos = sum(labels.values())
n_neg = len(labels) - n_pos
sources = by_split['train'] | by_split['test']
retention = len(sources) / N_CLIPS
two_class = sum(1 for v in frame_test.values() if 0 < sum(v) < len(v))

print(f'classes      : {defs}')
print(f'windows      : {len(windows)}  (train {len(labels)}, test {len(frame_test)})')
print(f'train classes: {n_pos} abnormal / {n_neg} normal   ratio {n_pos / max(n_neg, 1):.2f}:1')
print(f'sources kept : {len(sources)}/{N_CLIPS}  retention {retention:.3f}')
print(f'two-class test windows (the auc_macro population): {two_class}')
print(f'split        : {len(by_split["train"])} train / {len(by_split["test"])} test sources')

## 5. **Gate W** — the EDA profiler on the built corpus

`T2_SCORE_HEAD_KERNEL = 3` (set in §0): the default `constants.SCORE_HEAD_KERNEL = 9` spans
45 % of a 20-frame window and 56.2 % of a 16-frame one (lesson **C27**), so the coverage table
must be computed against the kernel the arms will actually train with — **and Phase 4 must
pass `model.score_head_kernel=3` explicitly**, because the package default is still 9.

**Also read the MIL line in §1.3 of the report.** `k = max(1, L // mil_topk_pct)` is **1 for
every window** at `L = 20` and the default `MIL_TOPK_PCT = 16`, i.e. `L_MIL` degenerates to a
plain max — one supervised frame per bag per step. Raising `W` does not fix that; only
lowering `loss.mil_topk_pct` does (`5` → k = 4 at W = 20). Decide it before Phase 4, not after.

In [ ]:
# T2_SCORE_HEAD_KERNEL is set in section 0 beside the window geometry -- it is part
# of the same decision and there is exactly one place to change it.
EDA = Path(os.environ['EDA'])
run([sys.executable, '-m', 'core.tools.eda', 'report',
     '--dataset', DATASET,
     '--data-dir', T2,
     '--clip-dir', CACHE,
     '--output-dir', EDA,
     '--score-head-kernel', T2_SCORE_HEAD_KERNEL,
     '--plots'], cwd=REPO)

In [ ]:
report = json.loads((EDA / constants.EDA_REPORT_JSON_FILENAME).read_text())
protocol, corpus_sec = report['protocol'], report['corpus']

leak    = protocol['clip_length_leak'].get('auc_clip_level')
oracle  = protocol['clip_constant_oracle']['auc_micro']
twoc    = protocol['macro_resolution']['two_class_clips']
cover   = corpus_sec['kernel_coverage_test']['median_coverage']
ratio   = n_pos / max(n_neg, 1)

CHECKS = [
    ('length leak (clip AUC)',      leak,      lambda v: v is not None and v <= 0.55, '<= 0.55'),
    ('clip oracle (micro AUC)',     oracle,    lambda v: v <= 0.75,                   '<= 0.75'),
    ('abnormal source retention',   retention, lambda v: v >= 0.90,                   '>= 0.90'),
    ('class ratio (abn:norm)',      ratio,     lambda v: 1 / 3 <= v <= 3,             'within 1:3'),
    (f'kernel coverage (median, k={T2_SCORE_HEAD_KERNEL})',
                                    cover,     lambda v: v <= 0.35,                   '<= 0.35'),
    ('two-class test windows',      twoc,      lambda v: v >= 300,                    '>= 300'),
]

print(f'{"criterion":34s} {"measured":>10s}  {"bar":>12s}  verdict')
print('-' * 76)
failed = []
for name, value, ok, bar in CHECKS:
    passed = ok(value)
    failed += [] if passed else [name]
    shown = 'skipped' if value is None else (f'{value:.4f}' if isinstance(value, float)
                                            else str(value))
    print(f'{name:34s} {shown:>10s}  {bar:>12s}  {"PASS" if passed else "FAIL"}')

# Which flag actually moves which criterion. The plan's risk 3 names
# --window-max-per-clip as THE lever; that is right for the class ratio and for the
# window count, and it is the wrong lever for the clip oracle -- the oracle follows the
# abnormal share of TEST FRAMES (C33's (F_norm + 0.5X)/(F_norm + X)), which the cap
# barely moves because it thins both classes at once. Raising --window-length is the
# lever there, and it is paid for in retention (C32). Re-measure BOTH, never one.
REMEDY = {
    'length leak (clip AUC)':
        'windows are fixed-length, so this should be 0.5 by construction -- a miss means '
        'the build did not window. Check section 4 before touching any flag.',
    'clip oracle (micro AUC)':
        'raise --window-length (more abnormal frames per abnormal bag). The per-clip cap '
        'is NOT the lever here. Raising W re-fires C32: re-check retention in the same run.',
    'abnormal source retention':
        'lower --window-length: a clip shorter than W contributes nothing (C32).',
    'class ratio (abn:norm)':
        'this is what --window-max-per-clip is for.',
    f'kernel coverage (median, k={T2_SCORE_HEAD_KERNEL})':
        'lower the kernel passed here AND model.score_head_kernel in Phase 4 (C27).',
    'two-class test windows':
        'raise --window-length, or lower --window-min-positive; both change what a '
        'positive bag is, so Gate W must be re-read whole.',
}

print()
if failed:
    print(f'GATE W FAILED on: {", ".join(failed)}')
    for name in failed:
        print(f'  - {name}: {REMEDY.get(name, "see plan section 4")}')
    print('STOP. Re-open plan section 4 and the parent plan section 5.2 table; the figures '
          'there are SIMULATIONS, and this cell is the measurement. Do not train on a '
          'corpus that failed.')
else:
    print('GATE W PASSED -- Phase 4 (training) is unblocked.')

print('\nEDA verdicts:')
for v in report['verdicts']:
    print(f"  [{v['level']}] {v['title']} -- {v['detail']}")

## 6. Record the gate (lesson **C17**) — in the **same cell** that measured it

Phase 1's per-clip frame census existed only on a VM that was later recycled, and it is gone.
Everything this run produced is copied to Drive here, before anything else runs.

The cell re-reads the census list off disk instead of reusing section 2's variable, so it also
works in the session where you reconnected and ran only §3–§6.

In [ ]:
import time

dest = Path(os.environ['EDA'])      # geometry-stamped; set once, in section 0
dest.mkdir(parents=True, exist_ok=True)

# The census is the expensive irreplaceable artifact: it is the only record of how many
# frames each clip had before the frames were deleted. Re-listed from disk, not taken
# from section 2 -- that variable does not exist after a reconnect.
census_files = sorted(p for p in counts_dir.glob('*.json') if p.name != RESOLVED.name)
census_dest = dest / 'counts'
census_dest.mkdir(exist_ok=True)
for c in census_files:
    shutil.copy2(c, census_dest / c.name)

proc = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
                      capture_output=True, text=True)
# `git rev-parse` printed EMPTY on the Drive mount during Phase 1 and the report
# recorded a blank commit. Say so explicitly rather than writing "".
commit = proc.stdout.strip() or f'UNKNOWN (git said: {proc.stderr.strip()!r})'

manifest = {
    'phase': 'Phase 2 (T2 corpus) + Phase 3 (Gate W)',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'commit': commit,
    'dataset': DATASET,
    'annotation': str(ANNOTATION),
    'annotated_clips': N_CLIPS,
    'data_dir': str(T2),
    'clip_cache': str(CACHE),
    # Read back from core.constants, so the manifest records what the build used and
    # not what a stale comment in this notebook claims (C17).
    'geometry': {'frame_stride': constants.FRAME_STRIDE,
                 'window_length': T2_WINDOW_LENGTH,
                 'window_stride': T2_WINDOW_STRIDE,
                 'window_max_per_clip': T2_MAX_PER_CLIP,
                 'window_min_positive': T2_MIN_POSITIVE,
                 'package_window_length': constants.DADA_ORIGIN_WINDOW_LENGTH,
                 'test_ratio': constants.DADA_ORIGIN_TEST_RATIO,
                 'seed': constants.SEED,
                 'score_head_kernel': T2_SCORE_HEAD_KERNEL},
    'shards': {'size': globals().get('SHARD_CLIPS'),
               'planned': len(globals().get('SHARDS', [])) or None,
               'censuses': len(census_files)},
    'census': {'usable_clips': len(usable), 'merged_clips': len(census)},
    'measured': {'windows': len(windows), 'train_windows': len(labels),
                 'test_windows': len(frame_test), 'train_abnormal': n_pos,
                 'train_normal': n_neg, 'sources_kept': len(sources),
                 'retention': retention, 'two_class_test_windows': two_class,
                 'length_leak': leak, 'clip_oracle': oracle,
                 'kernel_coverage_median': cover},
    'gate_w_failed': failed,
}
(dest / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

# meta.json is on Drive already (T2 is a Drive path) -- copied here too so the
# report folder is self-contained.
for name in (constants.META_FILENAME, constants.WINDOWS_FILENAME):
    shutil.copy2(T2 / name, dest / name)

print('recorded ->', dest)
for p in sorted(dest.rglob('*')):
    if p.is_file():
        print(f'   {p.relative_to(dest)}  ({p.stat().st_size / 1024:.1f} KiB)')
print(f'\ncommit: {commit}')
print('\nNext: plan section 6 -- Phase 4 arms, seeds 2024/2025/2026, KIP off/on, '
      'model.score_head_kernel=3, evaluated ZERO-SHOT on DoTA against the bar 0.6408. '
      'The in-domain T2 number is a sanity check only and never goes beside a published '
      'frame-level AUC (C8, C8b, C12).')